In [6]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, "..")

import pandas as pd
from src.data import load_run, audit_subjects, EXECUTED_RUNS, IMAGINED_RUNS
from src.config import RESULTS_DIR

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
for run in [4, 6]:
    raw = load_run(1, run)
    print(f"run {run}")
    for onset, desc in zip(raw.annotations.onset[:8], raw.annotations.description[:8]):
        print(f"  {onset:7.2f}  {desc}")

run 4
     0.00  T0
     4.20  T2
     8.30  T0
    12.50  T1
    16.60  T0
    20.80  T1
    24.90  T0
    29.10  T2
run 6
     0.00  T0
     4.20  T2
     8.30  T0
    12.50  T1
    16.60  T0
    20.80  T1
    24.90  T0
    29.10  T2


In [8]:
from mne.io import read_raw_edf
from mne.datasets import eegbci
from src.config import DATA_DIR

fname = eegbci.load_data(1, [4], path=DATA_DIR, update_path=False, verbose="ERROR")[0]
print(read_raw_edf(fname, preload=False, verbose="ERROR").ch_names[:12])
print(load_run(1, 4).ch_names[:12])

['Fc5.', 'Fc3.', 'Fc1.', 'Fcz.', 'Fc2.', 'Fc4.', 'Fc6.', 'C5..', 'C3..', 'C1..', 'Cz..', 'C2..']
['FC5', 'FC3', 'FC1', 'FCz', 'FC2', 'FC4', 'FC6', 'C5', 'C3', 'C1', 'Cz', 'C2']


In [9]:
subjects = range(1, 110)
df = pd.DataFrame(audit_subjects(subjects, EXECUTED_RUNS + IMAGINED_RUNS))
df.to_csv(RESULTS_DIR / "subject_audit.csv", index=False)
df.head(12)

,subject,run,family,sfreq,n_channels,duration_s,n_annotations,n_T0,n_T1,n_T2,other_labels,ch_names_hash,error
0,1,3,executed_lr,160.0,64,125.0,30,15,8,7,[],737393259880580152,None
1,1,7,executed_lr,160.0,64,125.0,30,15,8,7,[],737393259880580152,None
2,1,11,executed_lr,160.0,64,125.0,30,15,7,8,[],737393259880580152,None
3,1,4,imagined_lr,160.0,64,125.0,30,15,8,7,[],737393259880580152,None
4,1,8,imagined_lr,160.0,64,125.0,30,15,8,7,[],737393259880580152,None
5,1,12,imagined_lr,160.0,64,125.0,30,15,7,8,[],737393259880580152,None
6,2,3,executed_lr,160.0,64,123.0,30,15,8,7,[],737393259880580152,None
7,2,7,executed_lr,160.0,64,123.0,30,15,7,8,[],737393259880580152,None
8,2,11,executed_lr,160.0,64,123.0,30,15,8,7,[],737393259880580152,None
9,2,4,imagined_lr,160.0,64,123.0,30,15,7,8,[],737393259880580152,None


In [10]:
print("errors:", df[df.error.notna()][["subject", "run", "error"]].to_dict("records"))
print("sfreq:", df.sfreq.value_counts().to_dict())
print("n_channels:", df.n_channels.value_counts().to_dict())
print("montage consistent:", df.ch_names_hash.nunique() == 1)
print()
print(df.duration_s.describe())
print()
print(df.groupby("family")[["n_T0", "n_T1", "n_T2"]].describe().T)

errors: []
sfreq: {160.0: 636, 128.0: 18}
n_channels: {64: 654}
montage consistent: True

count    654.000000
mean     123.519878
std        2.487100
min      106.000000
25%      123.000000
50%      123.000000
75%      124.000000
max      181.000000
Name: duration_s, dtype: float64

family      executed_lr  imagined_lr
n_T0 count   327.000000   327.000000
     mean     15.067278    15.039755
     std       0.723234     0.622885
     min      12.000000    12.000000
     25%      15.000000    15.000000
     50%      15.000000    15.000000
     75%      15.000000    15.000000
     max      22.000000    19.000000
n_T1 count   327.000000   327.000000
     mean      7.556575     7.584098
     std       0.623427     0.568707
     min       6.000000     6.000000
     25%       7.000000     7.000000
     50%       8.000000     8.000000
     75%       8.000000     8.000000
     max      11.000000    10.000000
n_T2 count   327.000000   327.000000
     mean      7.510703     7.455657
     std     

In [11]:
#headroom check
rows = []
for s in range(1, 110):
    raw = load_run(s, 4)
    rows.append({
        "subject": s,
        "duration": round(raw.n_times / raw.info["sfreq"], 2),
        "last_onset": round(raw.annotations.onset[-1], 2),
        "headroom_s": round(raw.n_times / raw.info["sfreq"] - raw.annotations.onset[-1], 2),
    })
head = pd.DataFrame(rows)
print(head.headroom_s.describe())
head[head.headroom_s < 2.5]

count    109.000000
mean       4.294495
std        0.338525
min        4.000000
25%        4.100000
50%        4.100000
75%        4.600000
max        5.600000
Name: headroom_s, dtype: float64


,subject,duration,last_onset,headroom_s


In [12]:
subjects = range(1, 110)
df = pd.DataFrame(audit_subjects(subjects, EXECUTED_RUNS + IMAGINED_RUNS))
df.to_csv(RESULTS_DIR / "subject_audit.csv", index=False)

print("errors:", df[df.error.notna()][["subject", "run", "error"]].to_dict("records"))
print("sfreq:", df.sfreq.value_counts().to_dict())
print("n_channels:", df.n_channels.value_counts().to_dict())
print("montage consistent:", df.ch_names_hash.nunique() == 1)
print(df.duration_s.describe())

errors: []
sfreq: {160.0: 636, 128.0: 18}
n_channels: {64: 654}
montage consistent: True
count    654.000000
mean     123.519878
std        2.487100
min      106.000000
25%      123.000000
50%      123.000000
75%      124.000000
max      181.000000
Name: duration_s, dtype: float64


In [14]:

bad_sfreq = df[df.sfreq != 160.0]
print(bad_sfreq.groupby("subject")["run"].count())

bad_dur = df[(df.duration_s < 120) | (df.duration_s > 130)]
print(bad_dur.groupby("subject")["run"].agg(["count", list]))

print(df[df.sfreq != 160.0][["subject", "run", "sfreq", "duration_s", "n_T0", "n_T1", "n_T2"]])

subject
88     6
92     6
100    6
Name: run, dtype: int64
         count list
subject            
89           1  [3]
104          1  [8]
     subject  run  sfreq  duration_s  n_T0  n_T1  n_T2
522       88    3  128.0       124.0    19     9    10
523       88    7  128.0       124.0    19    10     9
524       88   11  128.0       124.0    19     9    10
525       88    4  128.0       124.0    19    10     9
526       88    8  128.0       124.0    19    10     9
527       88   12  128.0       124.0    19     9    10
546       92    3  128.0       124.0    19    10     9
547       92    7  128.0       124.0    19    10     9
548       92   11  128.0       124.0    19    10     9
549       92    4  128.0       124.0    19     9    10
550       92    8  128.0       124.0    19     9    10
551       92   12  128.0       124.0    19     9    10
594      100    3  128.0       123.0    12     6     6
595      100    7  128.0       123.0    12     6     6
596      100   11  128.0       123.0